In [ ]:
import os
import re
import asyncio
from dotenv import load_dotenv
from typing import List, Tuple

# LangChain & Specialized Imports
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader

# Custom modules (assuming these are your existing classes)
from src.ingestion.loader import DataLoader
from src.ingestion.chunker import DataStripping
from src.ingestion.embedder import EmbeddingGenerator
from src.retrieval.retriever import Retriever
from src.retrieval.evaluator_class import LoRAEvaluator
from src.db.vector_db import VectorStore

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_PATH = os.path.join(BASE_DIR, "data.txt")
ADAPTER_ZIP = os.path.join(BASE_DIR, "src", "retrieval", "rag_lora_adapter_zip.zip")
EXTRACT_PATH = os.path.join(BASE_DIR, "src", "retrieval", "rag_lora_adapter")

# -------------------------
# CRAG constants
# -------------------------
RELEVANCE_THRESHOLD = 1   # Documents with score >= 1 are considered relevant
MAX_WEB_PAGES = 3         # Number of web pages to fetch in fallback

# -------------------------
# Web fallback (async + non‑blocking)
# -------------------------
async def fetch_web_context_async(query: str) -> str:
    """Fetch relevant content from IBEX websites, runs in thread pool."""
    urls = [
        "https://ibex.co/",
        "https://waveix.ibex.co/",
        "https://www.ibex.co/industries/retail-ecommerce/",
        "https://www.ibex.co/staff-augmentation/"
    ]

    # Extract meaningful keywords (longer than 3 letters)
    keywords = {w.lower() for w in query.split() if len(w) > 3}
    if not keywords:
        keywords = {"ibex"}  # fallback if query is very short

    all_text = []

    for url in urls:
        try:
            # Run the synchronous WebBaseLoader in a thread
            loader = WebBaseLoader(url)
            docs = await asyncio.to_thread(loader.load)
            for doc in docs:
                content = doc.page_content.lower()
                # Include page only if it contains at least two keywords (to reduce noise)
                matched = sum(1 for kw in keywords if kw in content)
                if matched >= 2:
                    # Keep first 2000 chars per page (adjust as needed)
                    all_text.append(doc.page_content[:2000])
        except Exception as e:
            print(f"⚠️ Error fetching {url}: {e}")

    # Combine and truncate to a reasonable context length
    combined = "\n\n".join(all_text[:MAX_WEB_PAGES])
    return combined[:4000]  # Cap total length

# -------------------------
# Safe evaluator (sequential, thread‑safe)
# -------------------------
async def evaluate_docs_sequentially(evaluator, query: str, retrieved_docs: List[Document]) -> List[Tuple[Document, int]]:
    """
    Runs the LoRA evaluator one by one to avoid GPU thread conflicts.
    Returns list of (document, score) sorted by score descending.
    """
    scored = []
    for doc in retrieved_docs:
        # evaluate() returns a string like "1" or "Score: 2" – we'll extract reliably
        result = await asyncio.to_thread(evaluator.evaluate, query, doc.page_content)
        # Use a precise regex: look for the digit 0,1,2 as a whole word
        match = re.search(r"\b([0-2])\b", result)
        score = int(match.group(1)) if match else 0
        scored.append((doc, score))
    # Sort by score descending
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

# -------------------------
# Initialization
# -------------------------
def initialize_system():
    print("🚀 Initializing IBEX CRAG System...")

    # 1. Load and process your local knowledge base
    loader = DataLoader(DATA_PATH)
    raw_docs = loader.load()
    sentences = DataStripping(raw_docs).data_splitting()
    docs = [Document(page_content=s) for s in sentences]

    # 2. Build vector store and retriever
    embedding_model = EmbeddingGenerator(device="cpu").create_embeddings()
    vectorstore = VectorStore(docs, embedding_model).build()
    retriever = Retriever(vectorstore).get_retriever()

    # 3. Load the evaluator (LoRA model)
    evaluator = LoRAEvaluator("Qwen/Qwen2-0.5B", ADAPTER_ZIP, EXTRACT_PATH)

    # 4. LLM for final generation
    chat = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

    return retriever, evaluator, chat

# -------------------------
# Main CRAG Loop
# -------------------------
async def main():
    retriever, evaluator, chat = initialize_system()

    while True:
        query = input("\n[User]: ")
        if query.lower() in ["exit", "quit"]:
            break

        # ---- Step 1: Semantic retrieval ----
        retrieved_docs = retriever.invoke(query)

        # ---- Step 2: Relevance scoring (LoRA) ----
        scored_docs = await evaluate_docs_sequentially(evaluator, query, retrieved_docs)

        # ---- Step 3: Filter by threshold ----
        relevant_docs = [doc for doc, score in scored_docs if score >= RELEVANCE_THRESHOLD]

        # ---- Step 4: Corrective action if low confidence ----
        if not relevant_docs:
            print("🔍 Low relevance in local documents – performing web fallback...")
            web_context = await fetch_web_context_async(query)
            if web_context:
                # Use web content as the context (treat it as a single "document")
                context_text = web_context
            else:
                print("❌ No relevant information found in local or web sources.")
                continue  # Skip generation
        else:
            # Use top relevant documents (limit to 3)
            context_text = "\n\n".join([doc.page_content for doc in relevant_docs[:3]])

        # ---- Step 5: Generate final answer ----
        prompt = f"""You are the official IBEX AI Assistant.
Use the context below to answer accurately.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context_text}

Question: {query}
Answer:"""

        try:
            response = await chat.ainvoke(prompt)  # Use async version
            print(f"\n===== IBEX ASSISTANT =====\n{response.content}")
        except Exception as e:
            print(f"❌ Generation error: {e}")

if __name__ == "__main__":
    asyncio.run(main())